In [ ]:
import json
import numpy as np
import tensorflow as tf

In [ ]:
# Opening JSON file
f = open('/Users/adityanaik/Desktop/Summer project/medical-question-answer-data-master/icliniqQAs.json')
  
# returns JSON object as 
# a dictionary
data = json.load(f)

# Closing file
f.close()

#### Displaying one question and answer from the dataset

In [ ]:
data[0]['answer']

In [ ]:
data[0]['question']

##### Preprocessing
Removing the stop words and unnecessary punctuations to extract only useful unique words.

In [ ]:
from nltk.corpus import stopwords
from collections import defaultdict

stop = set(stopwords.words('english'))


In [ ]:
def CreateCorpus(text):
    corpus = []
    
    for x in text.split():
        corpus.append(x)
    return corpus



In [ ]:
test_words = []
for i in range(len(data)):
    test_words1 = CreateCorpus(data[i]['answer'])
    test_words2 = CreateCorpus(data[i]['question'])
    test_words = test_words + test_words1 + test_words2

In [ ]:
len(test_words)

In [ ]:
test_words

##### Removing punctuation marks

In [ ]:
# Removing punctuations
import string
def RemovePunct(text):
    table = str.maketrans('', '', string.punctuation)
    return text.translate(table)

In [ ]:
l = []
for word in test_words:
    l.append(RemovePunct(word))
test_words = l

##### Removing stopwords

In [ ]:
def RemoveStopwords(word_list):
    a = []
    for word in word_list:
        if word not in stop:
            a.append(word)
    return a

In [ ]:
b = RemoveStopwords(test_words)
test_words = b

In [ ]:
# Removing duplicates
c = set(test_words)
test_words = c

In [ ]:
len(test_words)

##### Total number of unique words from the iCliniQA dataset is 6233

## Test 1 Embedding generation ability
This test involves checking the ability fo the model to generate embedding for words. The model gets one point if it can and doesnt get a point if it cannot geneate an embedding. The accuracy is then measured by the ratio of the total number of points and the total number of words

### Evaluating GloVe

In [ ]:
# Loading the Glove file
import pandas as pd
import csv
words = pd.read_table('/Users/adityanaik/Desktop/Summer project/CODE files/glove.840B.300d.txt', sep=" ", index_col=0, header=None, quoting=csv.QUOTE_NONE)

In [ ]:
pos = 0
neg = 0
neg_words = []
for word in test_words:
    try:
        words.loc[word]
        pos += 1
    except:
        neg_words.append(word)
        neg +=1

In [ ]:
print('Number of words detected by GloVe', pos)
print('Number of words not detected by Glove ', neg)

In [ ]:
neg_words

##### Accuracy of the GloVe model is 92%

### Evaluating Word2Vec

In [ ]:
from gensim.models import Word2Vec, KeyedVectors
import csv

In [ ]:
model = KeyedVectors.load_word2vec_format('/Users/adityanaik/Desktop/Summer project/CODE files/GoogleNews-vectors-negative300.bin', binary = True, limit = 100000)

In [ ]:
pos = 0
neg = 0
neg_words = []
for word in test_words:
    try:
        model[word]
        pos += 1
    except:
        neg_words.append(word)
        neg +=1

In [ ]:
print('Number of words detected by Word2Vec', pos)
print('Number of words not detected by Word2Vec ', neg)

In [ ]:
neg_words

##### Accuracy of the Word2Vec model is 70%

### Evaluating Fasttext

In [ ]:
from gensim.models.fasttext import load_facebook_model

wv = load_facebook_model('/Users/adityanaik/Desktop/Summer project/CODE files/crawl-300d-2M-subword/crawl-300d-2M-subword.bin')

In [ ]:
pos = 0
neg = 0
neg_words = []
for word in test_words:
    try:
        wv.wv[word]
        pos += 1
    except:
        neg_words.append(word)
        neg +=1

In [ ]:
print('Number of words detected by fasttext', pos)
print('Number of words not detected by fasttext', neg)

In [ ]:
neg_words

##### Accuracy of the fasttext model is 100%

#### Measuring the performance on the symptom corpus extracted from the symptom-disease prediction dataset

In [ ]:
df = pd.read_csv(r'/Users/adityanaik/Desktop/Summer project/healthcare-chatbot-master/Data/Training.csv')

symptom_corpus = list(df.columns)
symptom_corpus = [x.replace('_', ' ') for x in symptom_corpus]
symptom_corpus = [x.translate(str.maketrans('', '', string.punctuation)) for x in symptom_corpus]
symptom_corpus = [x.translate(str.maketrans('', '', string.digits)) for x in symptom_corpus]
training_df = df
training_df.columns = symptom_corpus
symptom_corpus.remove('prognosis')



In [ ]:
symptom_corpus

In [ ]:
def GloVeEmbeddings(symptom_word, words):
    # List containing all the embeddings for each word in the sentence
    emb_arr = []
    # Representing the embeddings by taking the mean of each in case its
    for word in symptom_word.split():
        emb_arr.append(words.loc[word].to_numpy())
    emb_arr = np.array(emb_arr)
    fin_emb = np.mean(emb_arr, axis=0)
    return fin_emb

def FasttextEmbeddings(symptom_word, wv):
    # List containing all the embeddings for each word in the sentence
    emb_arr = []
    # Representing the embeddings by taking the mean of each in case its
    for word in symptom_word.split():
        emb_arr.append(wv.wv[word])
    emb_arr = np.array(emb_arr)
    fin_emb = np.mean(emb_arr, axis = 0)
    return fin_emb

def W2VEmbeddings(symptom_word):
    # List containing all the embeddings for each word in the sentence
    emb_arr = []
    # Representing the embeddings by taking the mean of each in case its
    for word in symptom_word.split():
        emb_arr.append(model[word])
    emb_arr = np.array(emb_arr)
    fin_emb = np.mean(emb_arr, axis = 0)
    return fin_emb

def similarityCheck(emb1, emb2):
    # Normalising both vectors
    normalize_emb1 = tf.nn.l2_normalize(emb1, 0)
    normalize_emb2 = tf.nn.l2_normalize(emb2, 0)
    # Calculating cos similarity
    cos_similarity = tf.reduce_sum(tf.multiply(normalize_emb1, normalize_emb2))
    return cos_similarity

### Evaluating the GloVe model for our symptom corpus

In [ ]:
pos = 0
neg = 0
neg_words = []
for word in symptom_corpus:
    try:
        GloVeEmbeddings(word)
        pos += 1
    except:
        neg_words.append(word)
        neg +=1

In [ ]:
pos

In [ ]:
neg

##### GloVe model can generate embeddings for 130 out of 132 symptom words

### Evaluating W2V model for our symptom corpus

In [ ]:
pos = 0
neg = 0
neg_words = []
for word in symptom_corpus:
    try:
        W2VEmbeddings(word)
        pos += 1
    except:
        neg_words.append(word)
        neg +=1

In [ ]:
pos

In [ ]:
neg

##### W2V model can only generate embedding for 95 our of 132 symptoms

### Evaluating the Fasttext model for our symptom corpus

In [ ]:
pos = 0
neg = 0
neg_words = []
for word in symptom_corpus:
    try:
        FasttextEmbeddings(word)
        pos += 1
    except:
        neg_words.append(word)
        neg +=1

In [ ]:
pos

In [ ]:
neg

##### Fasttext model can generate embeddings for all symptom words

## Test 2 synonym accuracy

In [ ]:
import pandas as pd

syn = pd.read_csv(r'/Users/adityanaik/Desktop/archive/synonyms.csv')
syn.head()

In [ ]:
a = list(syn['synonyms'][300:])
a

In [ ]:
b = []
c = []
for words in a:
    try:
        if len(words.split(";")) > 1:
            try:
                b.append(words.split(";")[0])
                c.append(words.split(";")[1])
            except:
                pass
    except:
        pass
    


##### Selecting 1000 words and their synonyms from the dataset

In [ ]:
main1 = b[0:1000]
main2 = c[0:1000]

### Evaluating the GloVe model for matching synonyms

In [ ]:

result = []
for i in range(len(main1)):
        if len(main1[i].split())==1 or len(main2[i].split())==1:
            try:
                emb1 = GloVeEmbeddings(main1[i], words).astype('float32')
                emb2 = GloVeEmbeddings(main2[i], words).astype('float32')
                cos = similarityCheck(emb1, emb2)
                if cos > 0.65:
                    result.append(1)
                else:
                    result.append(0)
            except:
                pass

            
            

In [ ]:
len(result)

In [ ]:
np.sum(result)

The GLoVe model could only generate embeddings for 672 words among the 1000 words out of which it could match only 156 words to its synonyms giving it an accuracy of 23%

### Evaluating the Fasttext model for matching synonyms

In [ ]:
result = []
for i in range(len(main1)):
        if len(main1[i].split())==1 or len(main2[i].split())==1:
            try:
                emb1 = FasttextEmbeddings(main1[i], wv).astype('float32')
                emb2 = FasttextEmbeddings(main2[i], wv).astype('float32')
                cos = similarityCheck(emb1, emb2)
                if cos > 0.65:
                    result.append(1)
                else:
                    result.append(0)
            except:
                pass

In [ ]:
len(result)

In [ ]:
np.sum(result)

The fasttext model could generate embeddings for 973 words out of 1000 out of which it could match the synonyms of 293 words giving it an accuracy of 30%

##### Based on the results of test 1 and test 2 we select the fasttext model to be used in our final implementation.